# Final Project Pembelajaran Mesin (F) - Kelompok 5
- Mario Napitupulu (5025241085)
- Willy Marcelius (5025241096)
- Rendy Tanuwijaya (5025241099)
- Hajendra Herlambang (5025241101)

Notebook ini diawali dengan tahap preprocessing untuk dataset Ames House Prices. Fokus preprocessing yang digunakan:

1. Feature engineering
2. Normalisasi fitur numerik
3. Encoding fitur kategorikal dengan ordinal encoding dan one-hot encoding

## 1. Preprocessing

### Import Library dan Load Dataset

In [21]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option('display.max_columns', 120)

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

### Identifikasi Missing Value

Pada dataset ini, beberapa nilai kosong muncul sebagai `NaN`. Untuk fitur seperti `Alley`, `PoolQC`, `Fence`, `FireplaceQu`, `Garage*`, dan `Bsmt*`, nilai kosong memiliki makna domain yaitu rumah tidak memiliki fasilitas tersebut. Karena itu, kolom-kolom tersebut akan diisi dengan kategori khusus `None`, bukan langsung dibuang.

In [22]:
missing_summary = (
    train_df.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame('jumlah_missing')
)
missing_summary['persentase'] = (missing_summary['jumlah_missing'] / len(train_df) * 100).round(2)
missing_summary

NameError: name 'train_df' is not defined

### Pemisahan Fitur dan Target

Kolom `SalePrice` digunakan sebagai target prediksi. Kolom `Id` tidak digunakan sebagai fitur karena hanya berfungsi sebagai identitas data.

In [23]:
TARGET = 'SalePrice'
ID_COL = 'Id'

X = train_df.drop(columns=[TARGET, ID_COL])
y = train_df[TARGET]
X_test_final = test_df.drop(columns=[ID_COL])

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test_final:', X_test_final.shape)

NameError: name 'train_df' is not defined

### Feature Engineering

Fitur baru dibuat dari kombinasi fitur asli agar model mendapat informasi yang lebih ringkas dan bermakna:

- `TotalSF`: total luas basement dan lantai rumah.
- `TotalBath`: total kamar mandi, dengan half bath diberi bobot 0.5.
- `TotalPorchSF`: total area teras/porch.
- `HouseAge`: umur rumah saat dijual.
- `RemodAge`: jarak tahun renovasi terakhir terhadap tahun penjualan.
- `GarageAge`: umur garasi saat rumah dijual.
- `HasPool`, `HasGarage`, `HasBsmt`, `HasFireplace`: fitur biner kepemilikan fasilitas.

In [ ]:
def add_engineered_features(df):
    """Menambahkan fitur turunan tanpa mengubah dataframe asli."""
    df = df.copy()

    df['TotalSF'] = df['TotalBsmtSF'].fillna(0) + df['1stFlrSF'].fillna(0) + df['2ndFlrSF'].fillna(0)
    df['TotalBath'] = (
        df['FullBath'].fillna(0)
        + 0.5 * df['HalfBath'].fillna(0)
        + df['BsmtFullBath'].fillna(0)
        + 0.5 * df['BsmtHalfBath'].fillna(0)
    )
    df['TotalPorchSF'] = (
        df['OpenPorchSF'].fillna(0)
        + df['EnclosedPorch'].fillna(0)
        + df['3SsnPorch'].fillna(0)
        + df['ScreenPorch'].fillna(0)
    )
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['GarageAge'] = df['YrSold'] - df['GarageYrBlt']
    df['GarageAge'] = df['GarageAge'].where(df['GarageYrBlt'].notna(), 0)

    df['HasPool'] = (df['PoolArea'].fillna(0) > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'].fillna(0) > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'].fillna(0) > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'].fillna(0) > 0).astype(int)

    return df


X_train_fe = add_engineered_features(X_train)
X_valid_fe = add_engineered_features(X_valid)
X_test_final_fe = add_engineered_features(X_test_final)

engineered_features = [
    'TotalSF', 'TotalBath', 'TotalPorchSF', 'HouseAge', 'RemodAge',
    'GarageAge', 'HasPool', 'HasGarage', 'HasBsmt', 'HasFireplace'
]
X_train_fe[engineered_features].head()

### Penentuan Kolom Encoding

Encoding dibedakan menjadi dua:

- **Ordinal encoding** untuk fitur yang memiliki urutan kualitas/kondisi, misalnya `Ex > Gd > TA > Fa > Po > None`.
- **One-hot encoding** untuk fitur nominal yang tidak memiliki urutan alami, misalnya `Neighborhood`, `RoofStyle`, dan `SaleCondition`.

In [ ]:
ordinal_features = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual',
    'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual',
    'GarageCond', 'PoolQC', 'Fence', 'LotShape', 'LandSlope',
    'PavedDrive', 'Utilities'
]

quality_order = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
exposure_order = ['None', 'No', 'Mn', 'Av', 'Gd']
finish_order = ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
functional_order = ['None', 'Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
garage_finish_order = ['None', 'Unf', 'RFn', 'Fin']
fence_order = ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
lot_shape_order = ['IR3', 'IR2', 'IR1', 'Reg']
land_slope_order = ['Sev', 'Mod', 'Gtl']
paved_drive_order = ['N', 'P', 'Y']
utilities_order = ['None', 'ELO', 'NoSeWa', 'NoSewr', 'AllPub']

ordinal_categories = [
    quality_order, quality_order, quality_order, quality_order, exposure_order,
    finish_order, finish_order, quality_order, quality_order,
    functional_order, quality_order, garage_finish_order, quality_order,
    quality_order, quality_order, fence_order, lot_shape_order, land_slope_order,
    paved_drive_order, utilities_order
]

categorical_features = X_train_fe.select_dtypes(include=['object', 'str']).columns.tolist()
onehot_features = [col for col in categorical_features if col not in ordinal_features]
numeric_features = X_train_fe.select_dtypes(exclude=['object', 'str']).columns.tolist()

print('Jumlah fitur numerik :', len(numeric_features))
print('Jumlah fitur ordinal :', len(ordinal_features))
print('Jumlah fitur nominal :', len(onehot_features))

### Pipeline Preprocessing

Pipeline ini melakukan:

1. Imputasi missing value numerik dengan median.
2. Normalisasi fitur numerik menggunakan `StandardScaler`.
3. Imputasi fitur ordinal dengan `None`, lalu ordinal encoding.
4. Imputasi fitur nominal dengan modus, lalu one-hot encoding.

Pipeline di-`fit` hanya pada data training untuk menghindari data leakage.

In [24]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

try:
    onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', onehot_encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, onehot_features)
    ],
    remainder='drop'
)

X_train_prepared = preprocessor.fit_transform(X_train_fe)
X_valid_prepared = preprocessor.transform(X_valid_fe)
X_test_final_prepared = preprocessor.transform(X_test_final_fe)

print('X_train_prepared:', X_train_prepared.shape)
print('X_valid_prepared:', X_valid_prepared.shape)
print('X_test_final_prepared:', X_test_final_prepared.shape)

NameError: name 'numeric_features' is not defined

### Hasil Preprocessing dalam Bentuk DataFrame

Cell berikut mengubah hasil preprocessing menjadi `DataFrame` agar nama fitur hasil normalisasi dan encoding bisa diperiksa. Data ini siap dipakai pada tahap training model.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names, index=X_train.index)
X_valid_prepared_df = pd.DataFrame(X_valid_prepared, columns=feature_names, index=X_valid.index)
X_test_final_prepared_df = pd.DataFrame(X_test_final_prepared, columns=feature_names, index=X_test_final.index)

X_train_prepared_df.head()

In [ ]:
preprocessing_report = pd.DataFrame({
    'tahap': [
        'Data awal training',
        'Setelah feature engineering',
        'Setelah normalisasi dan encoding'
    ],
    'jumlah_baris': [X_train.shape[0], X_train_fe.shape[0], X_train_prepared_df.shape[0]],
    'jumlah_fitur': [X_train.shape[1], X_train_fe.shape[1], X_train_prepared_df.shape[1]]
})

preprocessing_report

Sampai tahap ini, variabel yang siap digunakan untuk training model adalah:

- `X_train_prepared` atau `X_train_prepared_df`
- `X_valid_prepared` atau `X_valid_prepared_df`
- `y_train`
- `y_valid`

Untuk data testing akhir/submission, gunakan `X_test_final_prepared` atau `X_test_final_prepared_df`.

## 2. Training Model - Decision Tree Regressor (Willy Marcelius - 5025241096)

Bagian ini mengerjakan eksperimen model **Decision Tree Regressor** untuk memprediksi `SalePrice`. Eksperimen dibuat untuk membandingkan performa model pada kondisi normal dan pada dua skenario uji coba yang diusung kelompok:

1. **Feature Selection**: model hanya menggunakan sejumlah fitur terbaik berdasarkan skor `mutual_info_regression`.
2. **Label Price Logaritmik**: target `SalePrice` ditransformasi dengan `log1p` saat training, lalu prediksi dikembalikan ke skala harga asli dengan `expm1`.

Agar perbandingan lebih lengkap, eksperimen juga menyertakan kombinasi kedua skenario tersebut.

### Setup Eksperimen

Metrik evaluasi yang digunakan adalah:

- **RMSE**: menunjukkan rata-rata besar error dalam satuan harga rumah, dengan penalti lebih besar untuk error besar.
- **MAE**: menunjukkan rata-rata absolut error dalam satuan harga rumah.
- **RMSLE**: cocok untuk target harga karena mengukur error relatif pada skala logaritmik.
- **R2 Score**: menunjukkan proporsi variasi target yang mampu dijelaskan model.

Semua eksperimen menggunakan data training dan validation split yang sama dari tahap preprocessing.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from sklearn.tree import DecisionTreeRegressor

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    HAS_MATPLOTLIB = False
    print('matplotlib belum tersedia, sehingga cell grafik akan dilewati. Tabel performa tetap dapat digunakan.')

RANDOM_STATE = 42


def evaluate_regression(y_true, y_pred):
    """Menghitung metrik regresi pada skala harga asli."""
    y_pred_safe = np.clip(y_pred, 0, None)
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSLE': np.sqrt(mean_squared_log_error(y_true, y_pred_safe)),
        'R2': r2_score(y_true, y_pred)
    }


def get_candidate_k_values(n_features):
    """Menentukan pilihan jumlah fitur untuk skenario feature selection."""
    base_values = [20, 40, 60, 80, 120, 160]
    k_values = sorted({k for k in base_values if k < n_features})
    k_values.append('all')
    return k_values


param_grid = [
    {
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'max_features': max_features
    }
    for max_depth in [4, 6, 8, 10, None]
    for min_samples_split in [2, 10, 20]
    for min_samples_leaf in [1, 3, 5, 10]
    for max_features in [None, 'sqrt']
]

print('Jumlah kombinasi hyperparameter:', len(param_grid))
print('Jumlah fitur setelah preprocessing:', X_train_prepared_df.shape[1])

### Fungsi Training dan Evaluasi

Fungsi berikut menjalankan satu eksperimen Decision Tree Regressor. Jika `use_feature_selection=True`, jumlah fitur terbaik ikut dicari dari beberapa kandidat `k`. Jika `use_log_target=True`, model dilatih pada target `log1p(SalePrice)` dan hasil prediksi dikembalikan ke skala harga asli sebelum evaluasi.

In [ ]:
def run_decision_tree_experiment(name, use_feature_selection=False, use_log_target=False):
    y_train_model = np.log1p(y_train) if use_log_target else y_train
    selection_target = y_train_model if use_log_target else y_train

    k_candidates = get_candidate_k_values(X_train_prepared_df.shape[1]) if use_feature_selection else ['all']
    best_result = None

    for k in k_candidates:
        if use_feature_selection and k != 'all':
            selector = SelectKBest(
                score_func=lambda X_data, y_data: mutual_info_regression(
                    X_data,
                    y_data,
                    random_state=RANDOM_STATE
                ),
                k=k
            )
            X_train_model = selector.fit_transform(X_train_prepared_df, selection_target)
            X_valid_model = selector.transform(X_valid_prepared_df)
            selected_features = X_train_prepared_df.columns[selector.get_support()].tolist()
        else:
            selector = None
            X_train_model = X_train_prepared_df
            X_valid_model = X_valid_prepared_df
            selected_features = X_train_prepared_df.columns.tolist()

        for params in param_grid:
            model = DecisionTreeRegressor(random_state=RANDOM_STATE, **params)
            model.fit(X_train_model, y_train_model)
            y_pred_model = model.predict(X_valid_model)
            y_pred = np.expm1(y_pred_model) if use_log_target else y_pred_model

            metrics = evaluate_regression(y_valid, y_pred)
            result = {
                'Skenario': name,
                'Feature Selection': 'Ya' if use_feature_selection else 'Tidak',
                'Label Logaritmik': 'Ya' if use_log_target else 'Tidak',
                'Jumlah Fitur': len(selected_features),
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'RMSLE': metrics['RMSLE'],
                'R2': metrics['R2'],
                'Best Params': params,
                'Selected Features': selected_features,
                'Model': model,
                'Selector': selector,
                'Prediction': y_pred
            }

            if best_result is None or result['RMSE'] < best_result['RMSE']:
                best_result = result

    return best_result

### Tahap Training Model

Pada tahap ini, Decision Tree Regressor dilatih menggunakan beberapa konfigurasi skenario. Setiap skenario menggunakan data training yang sama, kemudian hasilnya dibandingkan pada validation set.

### Menjalankan Seluruh Skenario Uji Coba

Empat eksperimen berikut digunakan agar dampak setiap skenario dapat dibandingkan dengan jelas:

1. **Baseline**: tanpa feature selection dan tanpa transformasi log target.
2. **Feature Selection**: hanya menggunakan fitur terpilih.
3. **Label Logaritmik**: target harga ditransformasi logaritmik.
4. **Feature Selection + Label Logaritmik**: kombinasi kedua skenario.

In [ ]:
experiments = [
    ('Baseline Decision Tree', False, False),
    ('Feature Selection', True, False),
    ('Label Price Logaritmik', False, True),
    ('Feature Selection + Label Logaritmik', True, True)
]

experiment_results = []

for experiment_name, use_feature_selection, use_log_target in experiments:
    print(f'Menjalankan eksperimen: {experiment_name}')
    result = run_decision_tree_experiment(
        experiment_name,
        use_feature_selection=use_feature_selection,
        use_log_target=use_log_target
    )
    experiment_results.append(result)

print('Seluruh eksperimen selesai.')

### Tahap Testing dan Evaluasi pada Validation Set

Tahap testing dilakukan dengan memprediksi data validation (`X_valid_prepared_df`) yang tidak digunakan saat model dilatih. Hasil prediksi dibandingkan dengan `y_valid`, lalu dievaluasi menggunakan RMSE, MAE, RMSLE, dan R2.

### Tabel Hasil Performa Tiap Skenario

Tabel berikut menunjukkan performa terbaik dari masing-masing skenario berdasarkan pencarian hyperparameter sederhana. Model terbaik dipilih berdasarkan nilai **RMSE** terendah pada validation set.

In [ ]:
results_df = pd.DataFrame(experiment_results)

performance_columns = [
    'Skenario', 'Feature Selection', 'Label Logaritmik', 'Jumlah Fitur',
    'RMSE', 'MAE', 'RMSLE', 'R2', 'Best Params'
]

performance_table = results_df[performance_columns].copy()
performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']] = performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']].round(4)
performance_table = performance_table.sort_values('RMSE').reset_index(drop=True)
performance_table

### Visualisasi Perbandingan Performa

Grafik berikut membandingkan performa setiap skenario berdasarkan RMSE, MAE, RMSLE, dan R2. Untuk RMSE, MAE, dan RMSLE, nilai yang lebih kecil lebih baik. Untuk R2, nilai yang lebih besar lebih baik.

In [ ]:
plot_df = performance_table.set_index('Skenario')
metrics_to_plot = ['RMSE', 'MAE', 'RMSLE', 'R2']

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.ravel()

    for ax, metric in zip(axes, metrics_to_plot):
        plot_df[metric].plot(kind='bar', ax=ax, color='#4C78A8')
        ax.set_title(metric)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=25)
        ax.grid(axis='y', linestyle='--', alpha=0.35)

    plt.tight_layout()
    plt.show()
else:
    print('Grafik tidak ditampilkan karena matplotlib belum tersedia.')
    display(performance_table)

### Feature Selection: Fitur yang Terpilih

Cell berikut menampilkan fitur yang digunakan oleh skenario dengan feature selection. Daftar ini membantu mendokumentasikan fitur mana yang dianggap paling informatif untuk Decision Tree Regressor berdasarkan `mutual_info_regression`.

In [ ]:
feature_selection_rows = []

for result in experiment_results:
    if result['Feature Selection'] == 'Ya':
        for rank, feature_name in enumerate(result['Selected Features'], start=1):
            feature_selection_rows.append({
                'Skenario': result['Skenario'],
                'Rank': rank,
                'Fitur Terpilih': feature_name
            })

selected_features_df = pd.DataFrame(feature_selection_rows)
selected_features_df.head(30)

### Prediksi vs Nilai Aktual untuk Model Terbaik

Visualisasi berikut digunakan untuk melihat pola kesalahan model terbaik. Titik yang semakin dekat dengan garis diagonal menunjukkan prediksi yang semakin mendekati harga aktual.

In [25]:
best_result = min(experiment_results, key=lambda item: item['RMSE'])

if HAS_MATPLOTLIB:
    plt.figure(figsize=(7, 6))
    plt.scatter(y_valid, best_result['Prediction'], alpha=0.65, color='#2F855A')
    min_price = min(y_valid.min(), best_result['Prediction'].min())
    max_price = max(y_valid.max(), best_result['Prediction'].max())
    plt.plot([min_price, max_price], [min_price, max_price], color='#C53030', linestyle='--')
    plt.xlabel('SalePrice Aktual')
    plt.ylabel('SalePrice Prediksi')
    plt.title(f"Prediksi vs Aktual - {best_result['Skenario']}")
    plt.grid(linestyle='--', alpha=0.35)
    plt.show()
else:
    print('Scatter plot prediksi vs aktual tidak ditampilkan karena matplotlib belum tersedia.')

print('Model terbaik:', best_result['Skenario'])
print('RMSE terbaik :', round(best_result['RMSE'], 4))
print('Parameter    :', best_result['Best Params'])

ValueError: min() iterable argument is empty

### Pembahasan Hasil Uji Coba

Berdasarkan tabel dan grafik performa di atas, beberapa hal yang perlu diperhatikan adalah:

- **Baseline Decision Tree** menjadi pembanding utama karena model dilatih tanpa feature selection dan tanpa transformasi target. Performa baseline menunjukkan kemampuan awal Decision Tree setelah preprocessing.
- **Skenario Feature Selection** menguji apakah pengurangan fitur membantu model menghindari noise dari fitur hasil encoding yang jumlahnya cukup banyak. Jika RMSE dan RMSLE lebih rendah dari baseline, berarti fitur terpilih membantu model menjadi lebih stabil. Jika performanya turun, berarti sebagian informasi penting kemungkinan ikut terbuang saat seleksi fitur.
- **Skenario Label Price Logaritmik** menguji apakah transformasi target membantu model menangani distribusi harga rumah yang cenderung skewed. Jika RMSLE membaik, transformasi log membuat model lebih baik dalam menangkap perbedaan relatif antar harga rumah.
- **Skenario Kombinasi** menunjukkan apakah feature selection dan transformasi log saling melengkapi. Kombinasi ini tidak selalu otomatis menjadi yang terbaik karena Decision Tree sensitif terhadap pemilihan fitur dan kedalaman pohon.
- Model dengan **RMSE terendah** dipilih sebagai model Decision Tree Regressor terbaik pada validation set. Namun, metrik lain tetap perlu dilihat: MAE membantu membaca error rata-rata yang lebih mudah dipahami, RMSLE penting untuk konteks harga, dan R2 menunjukkan daya jelaskan model.

Secara umum, Decision Tree Regressor mudah diinterpretasikan, tetapi rentan overfitting. Karena itu, pembatasan seperti `max_depth`, `min_samples_split`, dan `min_samples_leaf` digunakan dalam eksperimen agar pohon tidak terlalu mengikuti data training secara berlebihan.

### Kesimpulan Bagian Decision Tree Regressor

Kesimpulan akhir untuk bagian ini dapat diisi berdasarkan baris terbaik pada `performance_table`. Skenario dengan RMSE terendah menjadi kandidat utama model Decision Tree Regressor terbaik, sedangkan perbandingan terhadap baseline menunjukkan apakah feature selection, label logaritmik, atau kombinasi keduanya benar-benar memberi peningkatan performa.

### Tahap Testing pada Data Test

Setelah skenario terbaik ditentukan dari validation set, konfigurasi terbaik tersebut digunakan kembali untuk melatih model pada seluruh data training. Langkah ini umum dilakukan sebelum membuat submission karena seluruh data berlabel dimanfaatkan untuk training akhir.

Pada tahap ini, model memprediksi `SalePrice` untuk `test.csv`. Karena data test Kaggle tidak memiliki label asli, evaluasi akhirnya diperoleh dari public score Kaggle setelah file submission diupload.

File submission dibuat mengikuti format `sample_submission.csv`, yaitu kolom `Id` dan `SalePrice`.

In [ ]:
from pathlib import Path

best_scenario = best_result['Skenario']
best_params = best_result['Best Params']
use_best_feature_selection = best_result['Feature Selection'] == 'Ya'
use_best_log_target = best_result['Label Logaritmik'] == 'Ya'

print('Skenario terbaik untuk submission:', best_scenario)
print('Menggunakan feature selection:', use_best_feature_selection)
print('Menggunakan label logaritmik:', use_best_log_target)
print('Parameter terbaik:', best_params)

In [ ]:
# Refit preprocessing pada seluruh data train agar model final memakai semua data berlabel.
final_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, onehot_features)
    ],
    remainder='drop'
)

X_full_fe = add_engineered_features(X)
X_test_submission_fe = add_engineered_features(X_test_final)

X_full_prepared = final_preprocessor.fit_transform(X_full_fe)
X_test_submission_prepared = final_preprocessor.transform(X_test_submission_fe)

final_feature_names = final_preprocessor.get_feature_names_out()
X_full_prepared_df = pd.DataFrame(X_full_prepared, columns=final_feature_names, index=X.index)
X_test_submission_prepared_df = pd.DataFrame(
    X_test_submission_prepared,
    columns=final_feature_names,
    index=X_test_final.index
)

print('X_full_prepared:', X_full_prepared_df.shape)
print('X_test_submission_prepared:', X_test_submission_prepared_df.shape)

In [ ]:
y_full_model = np.log1p(y) if use_best_log_target else y

if use_best_feature_selection:
    final_k = best_result['Selector'].k
    final_selector = SelectKBest(
        score_func=lambda X_data, y_data: mutual_info_regression(
            X_data,
            y_data,
            random_state=RANDOM_STATE
        ),
        k=final_k
    )
    final_X_train = final_selector.fit_transform(X_full_prepared_df, y_full_model)
    final_X_test = final_selector.transform(X_test_submission_prepared_df)
    final_selected_features = X_full_prepared_df.columns[final_selector.get_support()].tolist()
else:
    final_selector = None
    final_X_train = X_full_prepared_df
    final_X_test = X_test_submission_prepared_df
    final_selected_features = X_full_prepared_df.columns.tolist()

final_model = DecisionTreeRegressor(random_state=RANDOM_STATE, **best_params)
final_model.fit(final_X_train, y_full_model)

final_test_prediction_model = final_model.predict(final_X_test)
final_test_prediction = np.expm1(final_test_prediction_model) if use_best_log_target else final_test_prediction_model
final_test_prediction = np.clip(final_test_prediction, 0, None)

print('Jumlah fitur model final:', len(final_selected_features))
print('Contoh prediksi:', final_test_prediction[:5])

In [ ]:
sample_submission = pd.read_csv('sample_submission.csv')
submission = sample_submission.copy()
submission['SalePrice'] = final_test_prediction

submission_path = Path('submission_decision_tree_willy.csv')
submission.to_csv(submission_path, index=False)

submission.head()

## 3. Training Model - K-Nearest Neighbors Regressor (Mario Napitupulu - 5025241085)

Bagian ini mengerjakan eksperimen model **K-Nearest Neighbors (KNN) Regressor** untuk memprediksi `SalePrice`. Karena KNN menghitung jarak (*Euclidean/Manhattan distance*) antar data, dataset yang sudah dinormalisasi pada tahap preprocessing akan membuat algoritma ini bekerja optimal.

Sama seperti eksperimen sebelumnya, KNN akan diuji menggunakan empat skenario yang diusung kelompok:
1. **Baseline**: Menggunakan seluruh fitur yang sudah dinormalisasi.
2. **Feature Selection**: Memilih sejumlah fitur terbaik berdasarkan skor `mutual_info_regression`.
3. **Label Price Logaritmik**: Target `SalePrice` ditransformasi dengan `log1p` saat training.
4. **Feature Selection + Label Logaritmik**: Kombinasi kedua skenario.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Setup hyperparameter grid untuk KNN
knn_param_grid = [
    {
        'n_neighbors': n_neighbors,
        'weights': weights,
        'p': p
    }
    for n_neighbors in [3, 5, 7, 9, 11, 15]
    for weights in ['uniform', 'distance']
    for p in [1, 2] # p=1: Manhattan Distance, p=2: Euclidean Distance
]

print('Jumlah kombinasi hyperparameter KNN:', len(knn_param_grid))

In [26]:
def run_knn_experiment(name, use_feature_selection=False, use_log_target=False):
    y_train_model = np.log1p(y_train) if use_log_target else y_train
    selection_target = y_train_model if use_log_target else y_train

    # Meminjam fungsi get_candidate_k_values yang sudah didefinisikan sebelumnya
    k_candidates = get_candidate_k_values(X_train_prepared_df.shape[1]) if use_feature_selection else ['all']
    best_result = None

    for k in k_candidates:
        if use_feature_selection and k != 'all':
            selector = SelectKBest(
                score_func=lambda X_data, y_data: mutual_info_regression(
                    X_data,
                    y_data,
                    random_state=RANDOM_STATE
                ),
                k=k
            )
            X_train_model = selector.fit_transform(X_train_prepared_df, selection_target)
            X_valid_model = selector.transform(X_valid_prepared_df)
            selected_features = X_train_prepared_df.columns[selector.get_support()].tolist()
        else:
            selector = None
            X_train_model = X_train_prepared_df
            X_valid_model = X_valid_prepared_df
            selected_features = X_train_prepared_df.columns.tolist()

        for params in knn_param_grid:
            model = KNeighborsRegressor(**params)
            model.fit(X_train_model, y_train_model)
            y_pred_model = model.predict(X_valid_model)
            y_pred = np.expm1(y_pred_model) if use_log_target else y_pred_model

            metrics = evaluate_regression(y_valid, y_pred)
            result = {
                'Skenario': name,
                'Feature Selection': 'Ya' if use_feature_selection else 'Tidak',
                'Label Logaritmik': 'Ya' if use_log_target else 'Tidak',
                'Jumlah Fitur': len(selected_features),
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'RMSLE': metrics['RMSLE'],
                'R2': metrics['R2'],
                'Best Params': params,
                'Selected Features': selected_features,
                'Model': model,
                'Selector': selector,
                'Prediction': y_pred
            }

            if best_result is None or result['RMSE'] < best_result['RMSE']:
                best_result = result

    return best_result

In [27]:
knn_experiments = [
    ('Baseline KNN', False, False),
    ('Feature Selection', True, False),
    ('Label Price Logaritmik', False, True),
    ('Feature Selection + Label Logaritmik', True, True)
]

knn_experiment_results = []

for experiment_name, use_feature_selection, use_log_target in knn_experiments:
    print(f'Menjalankan eksperimen: {experiment_name}')
    result = run_knn_experiment(
        experiment_name,
        use_feature_selection=use_feature_selection,
        use_log_target=use_log_target
    )
    knn_experiment_results.append(result)

print('Seluruh eksperimen KNN selesai.')

Menjalankan eksperimen: Baseline KNN


NameError: name 'y_train' is not defined

### Tabel Hasil Performa Tiap Skenario (KNN)

Tabel berikut menunjukkan performa terbaik model KNN dari masing-masing skenario berdasarkan evaluasi pada validation set.

In [ ]:
import pandas as pd

knn_results_df = pd.DataFrame(knn_experiment_results)

performance_columns = ['Skenario', 'Feature Selection', 'Label Logaritmik', 'Jumlah Fitur', 'RMSE', 'MAE', 'RMSLE', 'R2', 'Best Params']
knn_performance_table = knn_results_df[performance_columns].copy()
knn_performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']] = knn_performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']].round(4)
knn_performance_table = knn_performance_table.sort_values('RMSE').reset_index(drop=True)

knn_performance_table